# Run-book 102-P4 — the fresh-clone dry run

**What this proves:** spec 102's end test, D17 (b) and AC14 (plus AC4). A fresh Claude Code session that
sees **only a clone of the repo** (no workspace `CLAUDE.md`, no memory, none of your Claude settings,
plugins or skills) takes a real `good first issue` (#122) to a pull request. **PASS:** green CI, all four
gates met, and **zero questions to you that the docs should have answered**. Each such question becomes a
doc fix, and that step is re-tried (the end of this notebook says how).

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs:** network (`git clone`, about 40 MB; `pip install`, about 1.3 GB on disk and 3–10 min, both
estimates); your usual `gh` and GitHub SSH login; a Claude login (the empty config folder asks for it
again); and **1–3 hours of your attention** while the session runs (estimate). You answer its permission
prompts and its questions.

**Order:** Steps 1–2 prepare the clone. Step 3 is the session itself, in a terminal. Steps 4–5 and the
Summary check the result. Steps 1–2's checks describe the clone *before* the session: after Step 3
starts, the agent changes branches and fills the config folder, so do not re-run them then.

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `fsd/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with fsd's dependencies.
- Open VS Code on the **`fsd/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `fsd/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed (e.g. VS Code was opened on the worktree folder, which has no `.venv` of its own)?
  **Select Another Kernel… → Python Environments → Enter interpreter path…** (wording varies by
  VS Code version) and give the main checkout's `fsd/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**Code — which `fsd` gets imported.** fsd is installed from the *main* checkout, so inside a
worktree a plain `import fsd` would quietly run `main`'s code, not the branch under test. The
Setup cell puts **this notebook's own** `src/` first on Python's import path and checks it worked.
(This is what the old run-books did with `export PYTHONPATH=src` — the same thing, invisibly.)

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running a cell resumes: downloads skip
files that are already complete. After a kernel restart, run **Setup** and **Settings** again,
then continue at any step — every step reads what it needs from disk.

**When you are done.** Paste back what the **Summary** cell says to paste (plus any failing cell's
message). Then **Restart & Clear All Outputs** and save: this file is tracked in git, and a test
refuses saved outputs.

In [ ]:
# Setup -- which Python, and which fsd. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be fsd's own virtual environment, fsd/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not fsd's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the fsd checkout that owns the venv

# 2. Which fsd? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "fsd").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import fsd

assert Path(fsd.__file__).is_relative_to(CHECKOUT / "src"), (
    f"fsd was already imported from {fsd.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("fsd    :", fsd.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else. The second block is **yours to fill
in during and after the session** (Step 3 says when); re-run this cell after each edit.

**Observation kinds** (one `dict` per event in `OBSERVATIONS`):
- `"question"`: the agent asked you something.
- `"stall"`: it got stuck, looped, or took a wrong path without asking (a question it did not ask).
- `"handoff"`: it handed you a run, as `AGENTS.md` rule 1 tells it to. Expected; not a failure.
- `"rule"`: it broke a rule in `AGENTS.md` or `CONTRIBUTING.md`.

`doc` is the file and section that should have answered it, or `"none"` if it was truly the maintainer's
call. `fix` is filled in later with the PR or issue that fixes the doc.

In [ ]:
# Settings
ATTEMPT = 1            # bump for each re-try after doc fixes merge; each attempt gets a fresh clone + config
REPO = "nikhilsrajan/fsd"
ISSUE = 122            # the `good first issue` the agent takes (labelled 2026-10-06)
CLONE = Path.home() / f"fsd-dryrun-{ATTEMPT}"                  # outside any folder with a CLAUDE.md
CLAUDE_CONFIG = Path.home() / f"fsd-dryrun-{ATTEMPT}-claude"   # an empty Claude Code config folder
PYTHON = "python3.11"  # the interpreter CONTRIBUTING.md's Setup names

# --- Filled in by you, the observer (Step 3). ---
AC4_AGENTS_MD_LISTED = None   # True / False: the new session's /context or /memory lists AGENTS.md
PR_NUMBER = None              # the PR the agent opened
FINDINGS_ALL_HANDLED = None   # True when every finding of the latest "## Gate-3 review" is fixed or filed
GATE4 = None                  # "n/a" if the PR touches no real data, cloud or pixels; "shown" if its evidence is in the PR
OBSERVATIONS = [
    # dict(kind="question", what="asked whether it may run pip", doc="AGENTS.md, Setup and checks", fix=""),
]

## Helpers

Two small helpers the steps share. Run this cell after Setup and Settings.

In [ ]:
import json
import shutil
import subprocess


def sh(*cmd, cwd=None):
    """Run a command; stop with the tail of its stderr if it fails; return its stdout."""
    r = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True)
    assert r.returncode == 0, f"{' '.join(map(str, cmd))} exited {r.returncode}:\n{r.stderr[-2000:]}"
    return r.stdout.strip()


def pr_gates(number):
    """Gates 1-3 as GitHub shows them for PR `number`, plus gate 4 from your Settings."""
    pr = json.loads(sh("gh", "pr", "view", number, "--repo", REPO, "--json",
                       "url,closingIssuesReferences,statusCheckRollup,comments"))
    checks = [c.get("conclusion") or c.get("state") for c in pr["statusCheckRollup"]]
    return {
        "url": pr["url"],
        "1_ci_green": bool(checks) and all(c in ("SUCCESS", "SKIPPED", "NEUTRAL") for c in checks),
        "2_links_issue": ISSUE in [i["number"] for i in pr["closingIssuesReferences"]],
        "3_reviewed": any(c["body"].startswith("## Gate-3 review") for c in pr["comments"])
                      and FINDINGS_ALL_HANDLED is True,
        "4_real_run": GATE4 in ("n/a", "shown"),
    }

## Step 1 — Make the clone

**What it does:** clones the repo from GitHub (`gh repo clone`) into `CLONE`, a folder in your home directory, outside the
workspace. Claude Code loads every `CLAUDE.md` in the folders *above* the one it starts in, so the
workspace's `CLAUDE.md` must not be above it.
**PASS if:** no `CLAUDE.md`, `CLAUDE.local.md` or `AGENTS.md` sits in any folder above the clone; the clone
is clean and at `origin/main`; its `CLAUDE.md` is exactly `@AGENTS.md` (AC4, first half); and the Claude
config folder does not exist yet or is empty (no settings, plugins, skills, memory or login from you).
**Time / disk:** under a minute; about 40 MB.

In [ ]:
# Step 1 -- clone (skipped if the clone already exists).
if not CLONE.exists():
    sh("gh", "repo", "clone", REPO, CLONE)   # uses your gh git protocol (SSH or HTTPS)

In [ ]:
# Step 1 -- check it.
above = [p / n for p in CLONE.parents for n in ("CLAUDE.md", "CLAUDE.local.md", "AGENTS.md") if (p / n).exists()]
assert not above, f"instruction files above the clone would load into the session: {above}"
assert sh("git", "status", "--porcelain", cwd=CLONE) == "", "the clone has local changes"
sh("git", "fetch", "origin", cwd=CLONE)
head, main = sh("git", "rev-parse", "HEAD", cwd=CLONE), sh("git", "rev-parse", "origin/main", cwd=CLONE)
assert head == main, f"the clone is at {head[:7]}, not origin/main {main[:7]}: bump ATTEMPT for a fresh clone"
assert (CLONE / "CLAUDE.md").read_text().strip() == "@AGENTS.md", "the clone's CLAUDE.md is not exactly @AGENTS.md"
assert not CLAUDE_CONFIG.exists() or not any(CLAUDE_CONFIG.iterdir()), (
    f"{CLAUDE_CONFIG} is not empty: an earlier session's config is in it. Bump ATTEMPT.")
print("clone at", head[:7], "= origin/main; nothing above it; config folder empty")

## Step 2 — Install the clone's `.venv` (`CONTRIBUTING.md`, Setup)

**What it does:** runs the install half of `CONTRIBUTING.md`'s Setup inside the clone. `AGENTS.md` says a
person does this before an agent session starts, because `pip` uses the network (its rule 1). The agent
runs the tests itself.
**PASS if:** the clone's own `.venv` imports `fsd` from the clone's `src/`, and the new `.venv` does not
show up in `git status`.
**Time / disk:** 3–10 min; about 1.3 GB (estimates).

In [ ]:
# Step 2 -- create the venv and install (skipped if it already exists).
venv_python = CLONE / ".venv" / "bin" / "python"
if not venv_python.exists():
    assert shutil.which(PYTHON), f"{PYTHON} is not on this kernel's PATH; CONTRIBUTING.md's Setup needs it"
    sh(PYTHON, "-m", "venv", ".venv", cwd=CLONE)
    sh(venv_python, "-m", "pip", "install", "-e", ".[dev,local]", cwd=CLONE)

In [ ]:
# Step 2 -- check it.
venv_python = CLONE / ".venv" / "bin" / "python"
where = sh(venv_python, "-c", "import fsd; print(fsd.__file__)", cwd=CLONE)
assert Path(where).is_relative_to(CLONE / "src"), f"the clone's .venv imports fsd from {where}"
assert sh("git", "status", "--porcelain", cwd=CLONE) == "", "the new .venv shows up in git status"
print("clone .venv imports", where)

## Step 3 — The session (you, in a terminal)

**What it does:** starts Claude Code in the clone with an **empty config folder**, so nothing of yours
loads: no settings, plugins, skills, hooks or memory. Run the cell below; it prints the exact lines.

1. Open a **new terminal** and paste the two printed command lines. `CLAUDE_CONFIG_DIR` is the only
   environment variable in this run-book: it points Claude Code at the empty folder, for this one
   command only. Log in and trust the folder when asked. Leave the model and every setting at its default.
2. Type `/context`. **AC4 PASS if** its memory files list `AGENTS.md` (if it shows only `CLAUDE.md`, try
   `/memory`). Set `AC4_AGENTS_MD_LISTED` in Settings.
3. Paste the printed prompt, exactly, and nothing else.
4. Play the maintainer:
   - Answer permission prompts as a careful maintainer would. They are not observations. Deny a push to
     `main` and any merge.
   - Answer each question briefly, and volunteer nothing else. Never mention the workspace, its
     `CLAUDE.md` or memory. Add one `OBSERVATIONS` entry per question.
   - Do not nudge. If it is stuck for about 10 minutes, or loops, record a `"stall"`, then give the
     smallest hint that unblocks it.
   - If it hands you a run, do it, paste the result back, and record a `"handoff"`.
5. The session ends when the agent says its PR meets the gates (it spawns its own reviewer, per
   `AGENTS.md`), or when you stop it. Fill in `PR_NUMBER`, `FINDINGS_ALL_HANDLED` and `GATE4`. Do not
   merge yet.

In [ ]:
# Step 3 -- print the lines to paste. Nothing runs here.
print("In a new terminal:")
print(f"  cd {CLONE}")
print(f"  CLAUDE_CONFIG_DIR={CLAUDE_CONFIG} claude")
print()
print("Then the prompt, exactly:")
print(f"  Take issue #{ISSUE} to a pull request.")

## Step 4 — Check the PR against the four gates

**What it does:** reads the PR from GitHub (gates 1–3) and your Settings (the review findings, gate 4).
**PASS if:** every check on the PR succeeded (1); the PR closes #122 (2); a `## Gate-3 review` comment
exists and you confirmed every finding is fixed or filed (3); and gate 4 is `"n/a"` or `"shown"` (4).

In [ ]:
# Step 4 -- check it.
assert PR_NUMBER, "set PR_NUMBER in Settings"
gates = pr_gates(PR_NUMBER)
print(json.dumps(gates, indent=2))
failed = [g for g, ok in gates.items() if g != "url" and not ok]
assert not failed, f"gates not met: {failed}"

## Step 5 — The D17 verdict (AC4, AC14)

**What it does:** checks your observer record.
**PASS if:** every observation has a known kind; AC4 held; and **no `"question"` or `"stall"` names a doc
that should have answered it.** A stall counts too: it is a question the agent did not ask.

**If it fails, re-try** (D17: each question becomes a doc fix, and that step is re-tried):
1. Post the Summary's Markdown on #102 (AC14 keeps every attempt's list).
2. Fix each gap in the named doc, in its own PR; put the PR number in `fix`. Merge them.
3. Bump `ATTEMPT`, empty `OBSERVATIONS`, and run Steps 1–3 again with the same prompt.
4. Stop the new session once it is past every step that had a gap. If such a step comes after the PR
   exists, close the earlier attempt's PR first (unmerged), so the agent starts clean.

In [ ]:
# Step 5 -- check it.
bad = [o for o in OBSERVATIONS if o.get("kind") not in ("question", "stall", "handoff", "rule")]
assert not bad, f"unknown observation kinds: {bad}"
assert AC4_AGENTS_MD_LISTED is True, "AC4: the session did not show AGENTS.md as loaded (Step 3.2)"
gaps = [o for o in OBSERVATIONS if o["kind"] in ("question", "stall") and o.get("doc", "none") != "none"]
for o in gaps:
    print(f"- {o['kind']}: {o['what']} -> {o['doc']}")
assert not gaps, f"{len(gaps)} doc gap(s): fix each, then re-try (see above)"

## Summary — paste this output back

Re-reads GitHub and your Settings, so it works after a restart (run Setup, Settings and Helpers first).
The Markdown part goes on #102 as is (AC14).

In [ ]:
gates = pr_gates(PR_NUMBER) if PR_NUMBER else {}
gaps = [o for o in OBSERVATIONS if o["kind"] in ("question", "stall") and o.get("doc", "none") != "none"]
kinds = ("question", "stall", "handoff", "rule")
summary = {
    "attempt": ATTEMPT,
    "pr": gates.get("url"),
    "gates": {g: ok for g, ok in gates.items() if g != "url"},
    "ac4_agents_md_listed": AC4_AGENTS_MD_LISTED,
    "observations": {k: sum(o["kind"] == k for o in OBSERVATIONS) for k in kinds},
    "doc_gaps": len(gaps),
    "d17_pass": bool(gates) and all(v for g, v in gates.items() if g != "url")
                and AC4_AGENTS_MD_LISTED is True and not gaps,
}
print(json.dumps(summary, indent=2, default=str))
print()
print(f"### D17 dry run, attempt {ATTEMPT}: PR {gates.get('url', '(none)')}")
print()
print("| # | kind | what happened | doc that should have answered | fix |")
print("|---|---|---|---|---|")
for i, o in enumerate(OBSERVATIONS, 1):
    print(f"| {i} | {o['kind']} | {o['what']} | {o.get('doc', 'none')} | {o.get('fix', '')} |")
print()
print(f"**D17 PASS:** {'yes' if summary['d17_pass'] else 'no'}")